<h1 style="color: #5439a7ff; text-align: center;">
  Practica 2: Sensado y análisis de video
</h1>
<h5 style="text-align: center;">
    Nombre del alumno: Jose Francisco Juarez Aceves 
    <br>
    Materia: Ciencia de Datos para Sensores Inteligentes
</h5>

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import pyrealsense2 as rs
from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

import joblib


<h2 style="color: #5439a7ff; text-align: center;">
  EXtraccion de caracteristicas
</h3>

In [11]:
def crear_pose_detector():

    base_options = python.BaseOptions(
        model_asset_path="pose_landmarker.task"
    )

    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1
    )

    detector = vision.PoseLandmarker.create_from_options(options)

    return detector


In [33]:
def extraer_keypoints_bag(ruta_bag, max_frames=None):

    detector = crear_pose_detector()
    keypoints_list = []

    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_device_from_file(ruta_bag, repeat_playback=False)
    config.enable_stream(rs.stream.color)

    pipeline.start(config)

    playback = pipeline.get_active_profile().get_device().as_playback()
    playback.set_real_time(False)

    frame_count = 0
    timestamp = 0

    try:
        while True:
            frames = pipeline.wait_for_frames()
            color_frame = frames.get_color_frame()

            if not color_frame:
                continue

            frame = np.asanyarray(color_frame.get_data())
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=frame_rgb
            )

            detection_result = detector.detect_for_video(
                mp_image,
                timestamp
            )

            timestamp += 1

            if detection_result.pose_landmarks:

                landmarks = detection_result.pose_landmarks[0]

                kp = np.array(
                    [[lm.x, lm.y, lm.z] for lm in landmarks],
                    dtype=np.float32
                )

                keypoints_list.append(kp)

            frame_count += 1

            if max_frames and frame_count >= max_frames:
                break

    except RuntimeError:
        pass

    pipeline.stop()

    if len(keypoints_list) == 0:
        return np.empty((0, 33, 3), dtype=np.float32)

    return np.stack(keypoints_list)


In [13]:
def keypoints_to_features_vectorizado(keypoints):

    if keypoints.shape[0] == 0:
        return np.empty((0, 0))

    # Centro entre hombros
    centro = (keypoints[:, 11] + keypoints[:, 12]) / 2
    keypoints_norm = keypoints - centro[:, np.newaxis, :]

    # Escalado
    dist_hombros = np.linalg.norm(
        keypoints[:, 11] - keypoints[:, 12],
        axis=1,
        keepdims=True
    )

    dist_hombros[dist_hombros == 0] = 1
    keypoints_norm /= dist_hombros[:, np.newaxis, :]

    # Distancias
    def distancia(p1, p2):
        return np.linalg.norm(
            keypoints_norm[:, p1] - keypoints_norm[:, p2],
            axis=1
        )

    distancias = np.stack([
        distancia(15, 16),
        distancia(13, 15),
        distancia(14, 16),
        distancia(11, 15),
        distancia(12, 16),
    ], axis=1)

    # Ángulos
    def calcular_angulo(a, b, c):
        ba = keypoints_norm[:, a] - keypoints_norm[:, b]
        bc = keypoints_norm[:, c] - keypoints_norm[:, b]

        cos_angle = np.sum(ba * bc, axis=1) / (
            np.linalg.norm(ba, axis=1) *
            np.linalg.norm(bc, axis=1) + 1e-6
        )

        cos_angle = np.clip(cos_angle, -1.0, 1.0)
        return np.arccos(cos_angle)

    angulos = np.stack([
        calcular_angulo(11, 13, 15),
        calcular_angulo(12, 14, 16),
        calcular_angulo(23, 25, 27),
        calcular_angulo(24, 26, 28),
    ], axis=1)

    # Velocidad
    velocidad = np.diff(keypoints_norm, axis=0)
    velocidad = np.pad(velocidad, ((1, 0), (0, 0), (0, 0)))

    velocidad_flat = velocidad.reshape(keypoints.shape[0], -1)
    posiciones_flat = keypoints_norm.reshape(keypoints.shape[0], -1)

    features = np.concatenate([
        posiciones_flat,
        distancias,
        angulos,
        velocidad_flat
    ], axis=1)

    return features.astype(np.float32)


<h2 style="color: #5439a7ff; text-align: center;">
  Creacion del dataset
</h3>

In [ ]:
ruta_base = "/home/josejuarez/Documents/Dataset/train"

features_totales = []
labels_totales = []

for persona in tqdm(os.listdir(ruta_base)):

    ruta_persona = os.path.join(ruta_base, persona)

    if os.path.isdir(ruta_persona):

        for archivo in os.listdir(ruta_persona):

            if archivo.endswith(".bag"):

                ruta_video = os.path.join(ruta_persona, archivo)

                keypoints = extraer_keypoints_bag(ruta_video)

                if keypoints.shape[0] == 0:
                    continue

                features = keypoints_to_features_vectorizado(keypoints)

                features_totales.append(features)
                labels_totales.extend([persona] * len(features))


  0%|          | 0/13 [00:00<?, ?it/s]INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1770873676.803146   50562 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770873676.999245   50562 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770873677.301465   50560 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
W0000 00:00:1770873717.144882   50599 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770873717.335226   50599 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling su

In [15]:
X = np.vstack(features_totales)
df_total = pd.DataFrame(X)
df_total["label"] = labels_totales


In [16]:
X = df_total.drop("label", axis=1).values
y = df_total["label"].values

le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [18]:
df_total.head()

,0,1,2,3,4,5,6,7,8,9,...,198,199,200,201,202,203,204,205,206,label
0,0.158436,-0.962360,-1.231681,0.201551,-1.090326,-1.293093,0.214655,-1.086776,-1.297966,0.224575,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Zarif
1,0.105984,-1.220627,-2.784247,0.160973,-1.383136,-2.768472,0.178270,-1.378756,-2.774738,0.191925,...,0.035355,1.334254,3.181245,0.091690,1.478231,2.659422,0.000630,1.475132,2.757558,Zarif
2,0.033195,-1.137125,-3.088992,0.085643,-1.287825,-2.939399,0.103372,-1.283842,-2.945008,0.118385,...,-0.123831,-0.576575,-1.482844,-0.121467,-0.509027,-1.424623,-0.140272,-0.508550,-1.416141,Zarif
3,-0.060754,-2.066563,-6.245658,0.034741,-2.341836,-5.753453,0.069322,-2.335057,-5.762342,0.099901,...,0.205909,5.206333,6.262162,0.420873,5.409673,5.647577,0.046773,5.442835,5.257819,Zarif
4,-0.172470,-2.312357,-6.823934,-0.066351,-2.622539,-6.172156,-0.024460,-2.614945,-6.180359,0.015135,...,-0.055009,1.263114,-0.344805,0.021593,1.400295,0.089393,-0.133255,1.424118,-0.614212,Zarif


<h2 style="color: #5439a7ff; text-align: center;">
  Entrenamiento de modelos
</h3>

In [19]:
modelo_rf = RandomForestClassifier(n_estimators=200, n_jobs=-1)
modelo_svm = SVC(kernel='rbf')
modelo_xgb = XGBClassifier(eval_metric='mlogloss')

modelos = {
    "Random Forest": modelo_rf,
    "SVM": modelo_svm,
    "XGBoost": modelo_xgb
}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)


In [21]:
resultados = {}

for nombre, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')  # puedes cambiar el average
    
    resultados[nombre] = {
        "accuracy": acc,
        "f1_score": f1
    }

    print("="*40)
    print(nombre)
    print("Accuracy:", acc)
    print("F1-score:", f1)
    print(classification_report(y_test, y_pred))



Random Forest
Accuracy: 0.9918822932521563
F1-score: 0.9918968091333349
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       130
           1       1.00      1.00      1.00        99
           2       0.99      0.99      0.99       197
           3       0.97      1.00      0.99       220
           4       1.00      0.98      0.99       127
           5       0.97      1.00      0.99       135
           6       1.00      0.99      1.00       105
           7       1.00      1.00      1.00       119
           8       0.99      0.99      0.99       198
           9       1.00      0.99      1.00       109
          10       1.00      0.99      0.99       171
          11       1.00      0.99      0.99       216
          12       1.00      0.99      0.99       145

    accuracy                           0.99      1971
   macro avg       0.99      0.99      0.99      1971
weighted avg       0.99      0.99      0.99      1971

SVM
Acc

In [30]:
mejor_modelo = modelo_xgb

<h2 style="color: #5439a7ff; text-align: center;">
  Prueba de mejor modelo
</h3>

In [34]:
ruta_test = "/home/josejuarez/Documents/Dataset/validacion_externa"

resultados = []

for archivo in tqdm(os.listdir(ruta_test)):

    if archivo.endswith(".bag"):

        ruta_video = os.path.join(ruta_test, archivo)
        keypoints = extraer_keypoints_bag(ruta_video)

        if keypoints.shape[0] == 0:
            continue

        features = keypoints_to_features_vectorizado(keypoints)

        if len(features) == 0:
            continue

        predicciones_cod = mejor_modelo.predict(features)
        predicciones = le.inverse_transform(predicciones_cod)
        serie_pred = pd.Series(predicciones)
        persona_predicha = serie_pred.value_counts().idxmax()
        confianza = serie_pred.value_counts(normalize=True).max()


        resultados.append({
            "Video": archivo,
            "Persona_predicha": persona_predicha,
            "Confianza_votacion": round(confianza, 3)
        })

  0%|          | 0/11 [00:00<?, ?it/s]W0000 00:00:1770878341.948793   54444 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770878342.159512   54447 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  9%|▉         | 1/11 [00:32<05:26, 32.70s/it]W0000 00:00:1770878374.586154   54469 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770878374.706747   54469 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
 18%|█▊        | 2/11 [01:11<05:27, 36.43s/it]W0000 00:00:1770878413.637075   54519 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. 

<h2 style="color: #5439a7ff; text-align: center;">
  Resultado de prediccion
</h3>

In [35]:
df_resultados = pd.DataFrame(resultados)
df_resultados


,Video,Persona_predicha,Confianza_votacion
0,grabacion_20260206_164242.bag,Diana Itzel,0.634
1,grabacion_20260206_163934.bag,Diana Itzel,0.686
2,grabacion_20260206_164123.bag,Edgar,0.505
3,grabacion_20260206_164315.bag,Diana Itzel,0.591
4,grabacion_20260206_164008.bag,Diana Itzel,0.432
5,grabacion_20260206_163749.bag,Diana Itzel,0.556
6,grabacion_20260206_163329.bag,Diana Itzel,0.633
7,grabacion_20260206_163238.bag,Diana Itzel,0.767
8,grabacion_20260206_164209.bag,Diana Itzel,0.725
9,grabacion_20260206_163516.bag,Diana Minerva,0.328


<h2 style="color: #5439a7ff; text-align: center;">
  Conclusion
</h3>

### El modelo demuestra fallos en detectar correctamente a los individuos en el dataset de "validacion_externa", esto lo atribuyo a la falta de normalizacion correcta de los datasets, ya que me falto recortar las partes donde mediapipe aun no detectaba nada y donde los sujetos dejaban de caminar. Asimismo, al revisar uno por uno los videos pasando por un algoritmo para ver los keypoints desplegados, pude notar varias cosas, como que tenia falsos positivos detectando puntos donde no habia una persona o en lugares aleatorios, siendo esto lo que tambien atrajo consecuencias al reconocimiento al entrenar los modelos, por lo que es obvio que me falto trabajar en el pre-procesamiento